# 01 · Base de conhecimento e busca
Monta a base (um trecho por artigo de lei, sempre com fonte), mede **recall@1** e **recall@3**
de três buscadores e escolhe o **limiar** de score abaixo do qual o sistema responde "não consta".

> O limiar é escolhido nas perguntas de **treino** (`avaliacao/lora_treino.jsonl`) e só depois
> conferido no **teste**, para não "colar" no conjunto de avaliação.

In [ ]:
# Preparação do ambiente (Colab ou local)
import os, sys, subprocess
REPO_URL = "https://github.com/SEU_USUARIO/assistente-acessibilidade-calcadas.git"  # <- troque pelo seu repositório

if "google.colab" in sys.modules:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", os.getcwd())

## 1. Base de conhecimento

In [ ]:
from src.base_conhecimento import coletar, montar, carregar_trechos, ler_fontes
from src.config import ARQ_TRECHOS
if not ARQ_TRECHOS.exists():      # a base já vem pronta no repositório; isto só refaz se faltar
    coletar(); montar()
trechos = carregar_trechos()
print(len(trechos), "trechos")
trechos[0]

In [ ]:
import pandas as pd
fontes = pd.DataFrame(ler_fontes())
n = pd.Series([t["fonte_id"] for t in trechos]).value_counts().rename("n_trechos")
tabela = fontes.set_index("id").join(n)[["titulo", "esfera", "formato", "licenca", "artigos", "n_trechos", "url"]]
tabela.to_csv("base/tabela_base.csv")
tabela

In [ ]:
import numpy as np
tam = np.array([len(t["texto"]) for t in trechos])
print(f"caracteres por trecho: mediana {np.median(tam):.0f}, p90 {np.percentile(tam, 90):.0f}, máx {tam.max()}")

## 2. Conjuntos de perguntas

In [ ]:
from src.avaliacao import carregar_jsonl
from src.config import ARQ_TESTE, ARQ_TREINO
teste, treino = carregar_jsonl(ARQ_TESTE), carregar_jsonl(ARQ_TREINO)
assert not {c["pergunta"] for c in teste} & {c["pergunta"] for c in treino}, "teste e treino se sobrepõem!"
ids = {t["id"] for t in trechos}
for c in teste + treino:
    assert set(c["trechos_gabarito"]) <= ids, (c["id"], c["trechos_gabarito"])
for nome, cj in [("teste", teste), ("treino", treino)]:
    print(nome, pd.Series([c["tipo"] for c in cj]).value_counts().to_dict())

## 3. Recall@1 e Recall@3 (conjunto de teste, perguntas que constam)

In [ ]:
from sentence_transformers import SentenceTransformer
from src.busca import Buscador, recall_at_k
from src.config import EMB_ID
emb = SentenceTransformer(EMB_ID)
buscadores = {m: Buscador(trechos, modo=m, modelo_emb=emb) for m in ["tfidf", "denso", "hibrido"]}
linhas = [{"buscador": m, "recall@1": recall_at_k(b, teste, 1), "recall@3": recall_at_k(b, teste, 3)}
          for m, b in buscadores.items()]
recall = pd.DataFrame(linhas).round(3)
recall.to_csv("avaliacao/busca_recall.csv", index=False)
recall

In [ ]:
# Perguntas em que o trecho certo NÃO veio no top-3 do buscador híbrido
b = buscadores["hibrido"]
for c in teste:
    if c["tipo"] != "consta": continue
    top = b.buscar(c["pergunta"], 3)
    if not {r["id"] for r in top} & set(c["trechos_gabarito"]):
        print(c["id"], c["pergunta"]); print("   esperado:", c["trechos_gabarito"]); print("   veio:", [r["id"] for r in top])

## 4. Limiar de "não consta" (escolhido no treino)

In [ ]:
import json, matplotlib.pyplot as plt
from src.busca import varrer_limiar, escolher_limiar
from src.config import ARQ_LIMIAR
varredura = pd.DataFrame(varrer_limiar(b, treino))
escolha = escolher_limiar(varredura.to_dict("records"))
print("limiar escolhido:", escolha)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(varredura.limiar, varredura.passa_respondiveis, label="respondíveis que passam")
ax.plot(varredura.limiar, varredura.barra_nao_consta, label='"não consta" barrados')
ax.plot(varredura.limiar, varredura.f1, "--", label="F1")
ax.axvline(escolha["limiar"], color="k", lw=1); ax.set_xlabel("score do 1º trecho"); ax.legend(); ax.set_title("Escolha do limiar (treino)")
plt.tight_layout(); plt.savefig("avaliacao/limiar.png", dpi=120); plt.show()

ARQ_LIMIAR.write_text(json.dumps({"limiar": escolha["limiar"], "buscador": "hibrido", **escolha}, indent=2))

In [ ]:
# Conferência no TESTE com o limiar fixado
sc = [(c["tipo"], b.buscar(c["pergunta"], 1)[0]["score"]) for c in teste]
lim = escolha["limiar"]
print(f'respondíveis que passam: {np.mean([s >= lim for t, s in sc if t == "consta"]):.1%}')
print(f'"não consta" barrados : {np.mean([s <  lim for t, s in sc if t == "nao_consta"]):.1%}')